# How Healthy and Equitable is Vancouver's Urban Tree Canopy?

*This is the exploratory draft for this project. The finished analysis is in `02-final-report.ipynb`.*

## Introduction

**Kaiya Sjoholm**

Vancouver is widely celebrated for its green spaces and natural beauty, but how healthy and equitable is the urban tree canopy that lines its streets? Street trees play a vital role in city life; they provide shade, improve air quality, support biodiversity, and contribute to the mental and physical wellbeing of residents. However, not all neighbourhoods may benefit equally from these benefits.

Using a subset of the City of Vancouver's open street trees dataset, containing 5,000 randomly sampled trees across 22 neighbourhoods, this analysis explores four key questions: 

- **Neighbourhood Variation:** Does tree size vary significantly across Vancouver's neighbourhoods, and what might drive those differences?
- **Trajectory:** Has Vancouver's tree planting activity changed over time, and where has it been concentrated?
- **Infrastructure:** Does the physical environment in which a tree is planted affect how large it grows?
- **Diversity:** How diverse is Vancouver's urban tree canopy, and are certain neighbourhoods at ecological risk from monoculture planning?

The dataset was obtained from the City of Vancouver's Open Data Portal and follows an Open Government License. It has been pre-cleaned and wrangled for use in this analysis. 

In [1]:
import altair as alt
import pandas as pd

vancouver_trees = pd.read_csv('../data/small_unique_vancouver.csv')

vancouver_trees.head()

,Unnamed: 0,std_street,on_street,species_name,neighbourhood_name,date_planted,diameter,street_side_name,genus_name,assigned,...,plant_area,curb,tree_id,common_name,height_range_id,on_street_block,cultivar_name,root_barrier,latitude,longitude
0,10747,W 20TH AV,W 20TH AV,PLATANOIDES,Riley Park,2000-02-23,28.5,EVEN,ACER,N,...,15,Y,21421,NORWAY MAPLE,4,0,NaN,N,49.252711,-123.106323
1,12573,W 18TH AV,W 18TH AV,CALLERYANA,Arbutus-Ridge,1992-02-04,6.0,ODD,PYRUS,N,...,7,Y,129645,CHANTICLEER PEAR,2,2300,CHANTICLEER,N,49.256350,-123.158709
2,29676,ROSS ST,ROSS ST,NIGRA,Sunset,NaN,12.0,ODD,PINUS,N,...,7,Y,154675,AUSTRIAN PINE,4,7800,NaN,N,49.213486,-123.083254
3,8856,DOMAN ST,DOMAN ST,AMERICANA,Killarney,1999-11-12,11.0,EVEN,FRAXINUS,N,...,7,Y,180803,AUTUMN APPLAUSE ASH,4,6900,AUTUMN APPLAUSE,N,49.220839,-123.036721
4,21098,EAST BOULEVARD,EAST BOULEVARD,HIPPOCASTANUM,Shaughnessy,NaN,15.5,ODD,AESCULUS,Y,...,N,Y,74364,COMMON HORSECHESTNUT,4,5200,NaN,N,49.238514,-123.154958


In [2]:
vancouver_trees.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Unnamed: 0          5000 non-null   int64  
 1   std_street          5000 non-null   str    
 2   on_street           5000 non-null   str    
 3   species_name        5000 non-null   str    
 4   neighbourhood_name  5000 non-null   str    
 5   date_planted        2363 non-null   str    
 6   diameter            5000 non-null   float64
 7   street_side_name    5000 non-null   str    
 8   genus_name          5000 non-null   str    
 9   assigned            5000 non-null   str    
 10  civic_number        5000 non-null   int64  
 11  plant_area          4950 non-null   str    
 12  curb                5000 non-null   str    
 13  tree_id             5000 non-null   int64  
 14  common_name         5000 non-null   str    
 15  height_range_id     5000 non-null   int64  
 16  on_street_block  

In [3]:
vancouver_trees.describe()

,Unnamed: 0,diameter,civic_number,tree_id,height_range_id,on_street_block,latitude,longitude
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.00000,5000.000000,5000.000000,5000.000000
mean,14861.920400,12.340888,2975.707600,128682.584600,2.73440,2960.227000,49.247349,-123.107128
std,8680.023278,9.266600,2078.580429,75412.260406,1.56957,2086.861052,0.021251,0.049137
min,2.000000,0.000000,2.000000,36.000000,0.00000,0.000000,49.202783,-123.220560
25%,7192.750000,4.000000,1300.500000,61321.500000,2.00000,1300.000000,49.230152,-123.144178
50%,14870.000000,10.000000,2639.000000,130130.500000,2.00000,2600.000000,49.247981,-123.105861
75%,22366.750000,18.000000,4123.000000,191332.000000,4.00000,4100.000000,49.263275,-123.063484
max,29992.000000,71.000000,9113.000000,270750.000000,9.00000,9100.000000,49.293930,-123.023311


## Observations from '.info()' and '.describe()'

**Missing Data:**
- 'date_planted' is the most significant gap, with only 2,363 out of 5,000 entries recorded. This limits time-based analysis but does not eliminate it.
- 'plant_area' only has 50 missing values and is otherwise complete.

**Data Quality:**
- 'diameter' has a minimum value of 0, which is likely a recording error. Trees with a diameter of 0 will be filtered out before any size-based analysis.
- 'diameter' also has a maximum of 71 inches compared to a mean of 12 and a 75th percentile of 18, suggesting a small number of very large outlier trees that may skew visualizations.

**Distribution:**
- 'height_range_id' has a mean of 2.7 and a max of 9, indicating most trees are relatively short with a few tall outliers.
- A minimum 'height_range_id' of 0 suggests some trees are in the 0-10ft range, likely very young or newly planted.

**Not Analytically Useful:**
- 'tree_id', 'civic_number', 'on_street_block', and 'Unnamed: 0' are identifiers or address details - their descriptive statistics are not meaningful for this analysis.

## Data Description

The dataset contains 5,000 Vancouver street trees with the following columns relevant to our questions:

- 'neighbourhood_name' - The neighbourhood the tree is located in
- 'diameter' - trunk diameter in inches, used as a proxy for tree maturity
- 'height_range_id' - height in ranges of 10ft (0 = 0-10 feet, 1 = 10-20 feet, etc.)
- 'date_planted' - date the tree was planted (note: many missing values)
- 'genus_name' / 'species_name' - tree taxonomy, used for diversity analysis
- 'plant_area' - planting context (boulevard, grate, cutout, etc.)

### Columns not used 
- 'std_street', 'on_street', 'civic_number', 'on_street_block' - address details not relevant to our questions
- 'cultivar_name', 'common_name', 'assigned', 'street_side_name' - too granular or not analytically useful
- 'root_barrier' - initially considered for infrastructure analysis but not included in final visualizations
- 'curb' - initially considered for infrastructure analysis but not included in final visualizations
- 'latitude' / 'longitude' - retained for potential geographic visualization but not used in this EDA
- 'tree_id' - unique identifier, not analytically useful

### Neighbourhood Variation: Do trees in wealthier neighbourhoods grow bigger?

To explore whether tree size varies across Vancouver neighbourhoods, we plot the mean trunk diameter for each neighbourhood sorted from largest to smallest. Diameter is used as a proxy for tree maturity, though we acknowledge this is imperfect as different species grow at different rates. 

In [4]:
neighbourhood_stats = (vancouver_trees.groupby('neighbourhood_name')[['diameter']]
                       .mean()
                       .reset_index())

neighbourhood_stats

,neighbourhood_name,diameter
0,Arbutus-Ridge,12.598571
1,Downtown,7.480117
2,Dunbar-Southlands,16.078115
3,Fairview,13.910821
4,Grandview-Woodland,12.603627
5,Hastings-Sunrise,12.185441
6,Kensington-Cedar Cottage,12.005600
7,Kerrisdale,13.904960
8,Killarney,10.030000
9,Kitsilano,15.080855


In [5]:
plot1 = alt.Chart(neighbourhood_stats).mark_bar().encode(
    y=alt.Y('neighbourhood_name:N', sort='-x', title='Neighbourhood'),
    x=alt.X('diameter:Q', title='Mean Diameter (inches)')
).properties(
    title='Mean Tree Size by Neighbourhood'
)

plot1

alt.Chart(...)

## Plot 1: Mean Tree Diameter by Neighbourhood

This chart shows the mean trunk diameter of street trees across Vancouver's neighbourhoods, sorted from largest to smallest. Diameter is used as a proxy for tree maturity, as older trees generally have wider trunks. However, this is an oversimplification; different genus and species types grow at fundamentally different rates, meaning diameter comparisons across neighbourhoods may reflect differences in species composition as much as differences in tree age or maturity.

**Key Observations:**
- Dunbar-Southlands, Kitsilano, and Shaughnessy have the largest mean diameter, while Downtown, Killarney, and Oakridge have the smallest.
- Neighbourhoods commonly associated with higher property values - such as West Point Grey, Shaughnessy, Dunbar-Southlands, Kitsilano, and Kerrisdale - all appear in the top half of the chart, suggesting a possible relationship between neighbourhood character and tree maturity. However, the pattern is not perfectly ordered, and the differences between many mid-ranking neighbourhoods are relatively small.

**Limitations & Next Questions:**
- This dataset does not include income or property value data, so any link to neighbourhood wealth is based on general knowledge rather than evidence. The variation in mean diameter across neighbourhoods is a real and interesting pattern, but its drivers, whether neighbourhood age, planting history, species selection, or socio-economic factors cannot be determined from this dataset alone. 
- An alternative explanation is neighbourhood age - older, more established neighbourhoods may simply have had more time for trees to grow, regardless of wealth.
- Species composition is also a significant confounding variable. A neighbourhood dominated by fast-growing ACER trees may show a higher mean diameter than one dominated by slower-growing ornamental PRUNUS trees, even if the trees are the same age. Controlling for species would be necessary for a more rigorous comparison.
- A more rigorous analysis would require linking this data to census income data or neighbourhood establishment dates, and species-specific growth rates.

*Note: Both diameter and height (height_range_id) were initially explored. They told a nearly identical story across neighbourhoods, confirming that diameter alone is a reliable relative indicator of tree size within this dataset, even if it cannot serve as a precise measure of tree age across species.*

### Trajectory: Is Vancouver's tree canopy getting younger over time? 

To explore planting trends over time, we plot the number of trees planted per year using only the 2,363 trees with a recorded planting date. A bar chart is used to show each year as a discrete count rather than implying a continuous trend. 

In [6]:
trees_by_year = (vancouver_trees.dropna(subset=['date_planted']).assign(
    year_planted=lambda x: pd.to_datetime(x['date_planted']).dt.year)
                 .groupby('year_planted')
                 .size()
                 .reset_index(name='tree_count'))

trees_by_year

,year_planted,tree_count
0,1989,9
1,1990,36
2,1991,15
3,1992,64
4,1993,80
5,1994,71
6,1995,97
7,1996,133
8,1997,111
9,1998,126


In [7]:
plot2a = alt.Chart(trees_by_year).mark_bar().encode(
    x=alt.X('year_planted:O', title='Year Planted'),
    y=alt.Y('tree_count:Q', title='Number of Trees Planted'),
).properties(title='Number of Trees Planted per Year in Vancouver')

plot2a

alt.Chart(...)

### Trajectory: Where was planting activity concentrated?

Building on the previous plot, we break down planting activity by neighbourhood to explore whether planting was evenly distributed across the city or concentrated in specific areas at specific times.

In [8]:
neighbourhood_trees_planted = (vancouver_trees.dropna(subset=['date_planted']).assign(
    year_planted=lambda x:pd.to_datetime(x['date_planted']).dt.year)
    .groupby(['year_planted', 'neighbourhood_name']).size().reset_index(name='tree_count'))

neighbourhood_trees_planted

,year_planted,neighbourhood_name,tree_count
0,1989,Arbutus-Ridge,3
1,1989,Dunbar-Southlands,2
2,1989,Grandview-Woodland,2
3,1989,Killarney,1
4,1989,Shaughnessy,1
...,...,...,...
522,2019,Shaughnessy,1
523,2019,Strathcona,1
524,2019,Sunset,2
525,2019,Victoria-Fraserview,2


In [9]:
plot2b = alt.Chart(neighbourhood_trees_planted).mark_rect().encode(
    x=alt.X('year_planted:O', title='Year Planted'),
    y=alt.Y('neighbourhood_name:N', title='Neighbourhood'),
    color=alt.Color('tree_count:Q', title='Number of Trees Planted')
).properties(title='Tree Planting Activity by Year and Neighbourhood')

plot2b

alt.Chart(...)

## Plots 2a & 2b: Tree Planting Trends Over Time

**Plot 2a** shows the total number of street trees planted per year across Vancouver, based on the 2,363 trees with a recorded planting date. Note that approximately 2,600 trees in the dataset have no recorded planting date and are excluded from this analysis. The trends observed here may not be fully representative of all street trees in Vancouver.

**Plot 2b** breaks this down further by neighbourhood, showing where planting activity was concentrated in any given year.

**Key Observations:**
- Tree planting activity was low in the late 1980s and early 1990s, then ramped up significantly through the mid-1990s, peaking in 1996, and remained relatively consistent through the 2000s and early 2010s before dropping sharply after 2014-2015.
- Planting was not evenly distributed across neighbourhoods. Rather than a city-wide effort happening uniformly, planting activity appears in concentrated bursts in specific neighbourhoods at specific times. This may reflect neighbourhood-level programs, responses to local tree loss, or neighbourhood revitalization efforts, such as urban renewal projects or infrastructure upgrades, which often include street tree planting as part of the investment package.
- Hastings-Sunrise, Renfrew-Collingwood, Sunset, and Victoria-Fraserview show some of the most intense planting activity. Interestingly, these are not the neighbourhoods that ranked highest for mean tree diameter in Plot 1, which may suggest that revitalization and replanting efforts have been directed toward neighbourhoods with historically lower tree coverage.

**Limitations & Next Questions:** 
- The sharp decline after 2014-2015 likely reflects a combination of genuinely reduced planting activity or incomplete data recording for recently planted trees. The dataset only extends to 2019, leaving seven years of planting history unaccounted for.
- The concentration of planting in certain neighbourhoods in certain years raises the question of whether this reflects city-led revitalization programs, responses to disease or construction, or other factors. Without additional context about city planning decisions, we cannot determine the cause from this dataset alone.
- The connection between recent planting activity and lower mean tree diameter suggests that newer planting efforts may be concentrated in neighbourhoods that historically had less tree coverage, a finding that adds nuance to our equity question from Plot 1 and warrants further investigation.

### Infrastructure: Does planting area type affect tree growth?

To explore whether physical planting infrastructure affects tree size, we compare the distribution of trunk diameter and height across different planting area types. Box plots are used to show the full distribution rather than just the mean.

In [10]:
def classify_plant_area(value):
    if pd.isna(value):
        return None
    try:
        float(value)
        return 'Boulevard'
    except ValueError:
        value_upper = value.strip().upper()
        if value_upper == 'B':
            return 'Behind Sidewalk'
        elif value_upper == 'G':
            return 'Grate'
        elif value_upper == 'N':
            return 'No Sidewalk'
        elif value_upper == 'C':
            return 'Cutout'
        else:
            return 'Other'

vancouver_trees = vancouver_trees.assign(plant_area_type=vancouver_trees['plant_area'].apply(classify_plant_area))

vancouver_trees['plant_area_type'].value_counts()

plant_area_type
Boulevard          3838
No Sidewalk         469
Behind Sidewalk     310
Cutout              256
Grate                62
Other                15
Name: count, dtype: int64

In [11]:
trees_plant_area = vancouver_trees.dropna(subset=['plant_area_type']).rename(columns={'diameter': 'Diameter (inches)', 'height_range_id': 'Height (10ft increments)'})

plot3 = alt.Chart(trees_plant_area).mark_boxplot().encode(
    x=alt.X('plant_area_type:N', title='Planting Area Type'),
    y=alt.Y(alt.repeat('repeat'), type='quantitative')
).repeat(repeat=['Diameter (inches)', 'Height (10ft increments)'],
         columns=2).properties(title='Tree Size Distribution by Planting Area Type')

plot3

alt.RepeatChart(...)

## Plot 3: Tree Size Distribution by Planting Area Type

These box plots show the distribution of trunk diameter and height across different planting area types, exploring whether the physical infrastructure a tree is planted in affects how large it grows.

**Key Observations:**
- **Diameter is more affected by planting area type than height**. The diameter chart shows meaningful differences in both median and spread across categories, while the height chart shows similar interquartile ranges across all planting types.
- **Behind Sidewalk trees have the highest median diameter**, suggesting that trees with more soil access and root space tend to develop wider trunks over time.
- **Boulevard trees have the most outliers in diameter**, including some very large trees exceeding 70 inches. This likely reflects older trees planted before modern urban infrastructure constraints were introduced.
- **Grate and Cutout trees tend to have smaller diameters**, consistent with the idea that physically constrained root systems limit trunk growth. However, their height distributions are similar to less constrained categories, suggesting trees can grow tall even when their roots are restricted.
- **No Sidewalk trees have a notably higher median diameter** compared to Grate and Cutout, further supporting the idea that root access plays a role in trunk development.

**Limitations & Next Questions:**
- **Tree age is a significant confounding variable in this analysis**. A tree planted in a grate in 1990 will likely have a larger diameter than one planted in 2015, regardless of planting infrastructure. Without complete planting date data for all trees, we cannot fully account for this effect. Any conclusions about the impact of planting infrastructure on tree size should be treated with caution until age is controlled for.
- **Tree species is an equally important confounding variable.** Cities often deliberately match species to planting environments. Smaller ornamental cherry blossoms are commonly planted in grates and cutouts, while large shade trees like oaks and maples are more typically planted in boulevards where root space is available. This means the size differences we observe across planting area types may reflect deliberate species selection as much as, or more than, the physical effect of root restriction. Distinguishing these two effects would require controlling for species in the analysis.
- Planting area type is also correlated with neighbourhood, making it difficult to isolate the effect of infrastructure alone. Older neighbourhoods may have more Boulevard trees simply because that was the dominant planting method historically.
- The "Other" category contains only 15 trees with unclassified planting codes and should be interpreted with caution.
- A more rigorous analysis would control for tree age and species to isolate the true effect of planting infrastructure on tree growth.

### Diversity: Which neighbourhoods have the most monocultures?

To explore genus diversity across Vancouver neighbourhoods, we plot the number of unique genera per neighbourhood, coloured by the dominant genus. This allows us to assess both the breadth of diversity and whether any single genus dominates in ways that could pose an ecological risk.

In [12]:
# Number of unique genera per neighbourhood
genus_diversity = (vancouver_trees.groupby('neighbourhood_name')['genus_name'].nunique().reset_index().rename(columns={'genus_name': 'unique_genera'}))

# Dominant genus per neighbourhood
dominant_genus = (vancouver_trees.groupby(['neighbourhood_name', 'genus_name']).size().reset_index(name='count').sort_values('count', ascending=False).groupby('neighbourhood_name').first().reset_index().rename(columns={'genus_name': 'dominant_genus'}))

# Merge together
diversity_stats = genus_diversity.merge(dominant_genus[['neighbourhood_name', 'dominant_genus']], on='neighbourhood_name')

diversity_stats

,neighbourhood_name,unique_genera,dominant_genus
0,Arbutus-Ridge,30,PRUNUS
1,Downtown,27,ACER
2,Dunbar-Southlands,36,PRUNUS
3,Fairview,26,ACER
4,Grandview-Woodland,26,ACER
5,Hastings-Sunrise,37,ACER
6,Kensington-Cedar Cottage,35,ACER
7,Kerrisdale,29,ACER
8,Killarney,32,PRUNUS
9,Kitsilano,36,ACER


In [13]:
plot4 = alt.Chart(diversity_stats).mark_bar().encode(
    y=alt.Y('neighbourhood_name:N', sort='-x', title='Neighbourhood'),
    x=alt.X('unique_genera:Q', title='Number of Unique Tree Genera'),
    color=alt.Color('dominant_genus:N', title='Dominant Genus')
).properties(title='Tree Genus Diversity by Neighbourhood')

plot4

alt.Chart(...)

## Plot 4: Tree Genus Diversity by Neighbourhood

This chart shows the number of unique tree genera per neighbourhood, coloured by the dominant genus, to explore whether Vancouver's urban tree canopy is ecologically diverse or concentrated in a small number of species.

**Key Observations:**
- There is meaningful diversity across neighbourhoods, even Strathcona, the least diverse neighbourhood, has 19 unique genera. Renfrew-Collingwood leads with 46, more than twice Strathcona's count.
- Renfrew-Collingwood's high diversity is interesting given that it was also one of the most actively planted neighbourhoods in Plot 2b, suggesting that recent planting efforts may have intentionally introduced a wider variety of species.
- Despite this surface-level diversity, every single neighbourhood is dominated by either ACER (maple) or PRUNUS (cherry/plum), no exceptions. Vancouver is effectively a two-genus city at the street tree level.
- ACER dominates more neighbourhoods overall, while PRUNUS tends to dominate in neighbourhoods like Dunbar-Southlands that ranked highly for mean tree diameter in Plot 1. This raises an interesting question about whether tree species selection reflects neighbourhood character or investment, wealthier neighbourhoods appear to have both larger trees and a higher prevalence of ornamental PRUNUS species, potentially connecting back to the cultural significance of cherry blossoms in certain parts of Vancouver.
- This concentration in two genera represents a significant ecological risk. If either ACER or PRUNUS were to face a widespread disease or pest outbreak, a large portion of Vancouver's tree canopy could be affected simultaneously.

**Limitations & Next Questions:**
- It is worth noting that this analysis is based on a random subset of 5,000 trees from the full Vancouver street trees dataset. The dominant genus findings may not perfectly represent the true genus composition of each neighbourhood, and results should be interpreted with that caveat in mind.
- This analysis is at the genus level; diversity at the species level may tell a different story, as PRUNUS alone encompasses cherry, plum, and peach trees among others.
- A more complete risk assessment would look at the percentage of trees belonging to the dominant genus in each neighbourhood, rather than just identifying which genus dominates.
- The connection between PRUNUS dominance and higher mean tree diameter in wealthier neighbourhoods warrants further investigation; it is unclear whether this reflects deliberate planting choices, historical patterns, or cultural preferences for ornamental species in certain parts of the city. 

## Concluding Remarks

This exploratory analysis investigated the health and equity of Vancouver's tree canopy through four lenses: neighbourhood variation, trajectory, infrastructure, and diversity. The following four visualizations will be carried forward into the final report:

**1. Mean Tree Diameter by Neighbourhood (Plot 1 - Neighbourhood Variation)**
This plot will be refined for the final report to tell a cleaner story about neighbourhood-level variation in tree size. Rather than framing this as an equity story, which would require income or property value data we do not have, the final report will present it as a pattern worth investigating further. Tree size varies significantly across Vancouver's neighbourhoods, with some neighbourhoods having mean diameters more than twice those of others. The drivers of this variation, whether neighbourhood age, planting history, species selection, or socio-economic factors, cannot be determined from this dataset alone. It is also important to note that diameter comparisons across neighbourhoods may reflect differences in species composition as much as differences in tree maturity, since different types grow at fundamentally different rates. The final version will include a clearer title that reflects this framing and a note contextualizing these limitations for the audience.

**2. Tree Planting Trends Over Time (Plots 2a & 2b - Trajectory)**
These two plots will be presented as a pair in the final report. Plot 2a will be refined with a reference line marking the point of decline after 2014-2015, and Plot 2b will include a clearer title and legend explanation to help the audience read the heat map. Both plots will include a prominent note that approximately 2,600 trees have no recorded planting date and are excluded from the analysis, and that the post-2015 decline likely reflects incomplete data recording as much as a genuine decline in planting activity.

**3. Tree Size Distribution by Planting Area Type (Plot 3 - Infrastructure)**
For the final report, the two box plots will be refined with cleaner axis labels and a note explaining that tree age and species are significant confounding variables that limit causal conclusions. Critically, diameter differences across planting area types may reflect deliberate species selection for different environments as much as the physical effect of root restriction, and since different species grow at different rates, diameter alone cannot confirm that one planting type produces larger or more mature trees than another. The audience will also need context about what each planting area type means in practice. For example, a grate is a metal grid in a sidewalk, and a cutout is a small opening in pavement, as these terms may not be immediately intuitive to a general audience.

**4. Tree Genus Diversity by Neighbourhood (Plot 4 - Diversity)**
This plot will be refined for the final report by adding a percentage label showing what proportion of each neighbourhood's trees belong to the dominant genus, making the monoculture risk more concrete and quantifiable. A brief explanation of what ACER (maple) and PRUNUS (cherry/plum) are in plain language will be added for audiences unfamiliar with botanical genus names. The ecological risk of a two-genus dominant canopy will be clearly communicated as the central finding of this section.

**Data Context for the Audience**
- This analysis is based on a random subset of 5,000 trees from the city of Vancouver's full street trees dataset, and findings may not perfectly represent the complete canopy.
- Approximately 2,600 trees have no recorded planting date, limiting the scope of time-based analysis.
- Diameter is used throughout as a proxy for tree maturity, but this is an oversimplification. Different genus and species types grow at fundamentally different rates; a 10-inch diameter PRUNUS tree and a 10-inch diameter ACER tree are not necessarily the same age. Neighbourhood comparisons based on diameter may therefore reflect differences in species composition as much as differences in tree age or maturity. A more rigorous analysis would control for species when using diameter as a proxy for age.
- No income, property value, or neighbourhood establishment date data were available, meaning any socio-economic interpretations of neighbourhood-level patterns should be treated as hypotheses for further investigation rather than confirmed findings. 

## References

- City of Vancouver Open Data Portal — [Street Trees Dataset](https://opendata.vancouver.ca/explore/dataset/public-trees/information/?disjunctive.common_name&disjunctive.species_name). Licensed under the Open Government License – Vancouver.
- Course Modules - Data Visualization, UBC Key Capabilities in Data Science